# 16 - Hybrid Search
---

In the previous notebook, we learned Qdrant.

Qdrant performs semantic search using vector embeddings.

But semantic search is not always enough.

Sometimes users search for:

- Product IDs
- Company names
- Error codes
- Version numbers

These are better handled by keyword search.

Hybrid Search combines both approaches to improve retrieval quality.

##  History

Before embedding models,

search engines relied mainly on keyword matching.

Algorithms such as BM25 ranked documents based on matching words.

Later,

vector embeddings enabled semantic search.

Researchers discovered that combining

keyword search

and

semantic search

often produced better retrieval results.

This became Hybrid Search.

## Think Like a Researcher

Imagine a user asks:

"NVIDIA Q1 2024 revenue"

Keyword Search

↓

Finds documents containing the exact words.

Semantic Search

↓

Finds documents discussing the same topic even if different words are used.

Using both methods together often gives better results.

In [1]:
documents = [
    "NVIDIA reported record revenue in Q1 2024.",
    "Apple released a new iPhone.",
    "The company's quarterly earnings increased.",
    "GPU demand continues to grow."
]

query = "NVIDIA Q1 revenue"

In [3]:
#Keyword search
results = []

for doc in documents:

    score = sum(
        word.lower() in doc.lower()
        for word in query.split()
    )

    results.append((doc, score))

sorted(results, key=lambda x: x[1], reverse=True)

[('NVIDIA reported record revenue in Q1 2024.', 3),
 ('Apple released a new iPhone.', 0),
 ("The company's quarterly earnings increased.", 0),
 ('GPU demand continues to grow.', 0)]

## Observation

Keyword search rewards documents that contain the exact query terms.

However,

it may miss documents that discuss the same concept using different wording.

In [4]:
#Semantic search
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = model.encode(documents)

query_embedding = model.encode([query])

scores = cosine_similarity(
    query_embedding,
    embeddings
)

scores

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

array([[0.8301021 , 0.10419019, 0.31918585, 0.37041822]], dtype=float32)

## Observation

Semantic search compares meaning rather than exact words.

It can retrieve relevant documents even when the wording differs.

## Why Hybrid Search?

Keyword Search

↓

Excellent for exact terms.

Semantic Search

↓

Excellent for meaning.

Hybrid Search

↓

Uses the strengths of both.

## Hybrid Search Pipeline

```
User Query

↓

Keyword Search (BM25)

↓

Semantic Search (Embeddings)

↓

Combine Results

↓

Return Top K
```

## Reciprocal Rank Fusion (RRF)

Hybrid Search often combines ranking results using

**Reciprocal Rank Fusion (RRF)**.

Instead of averaging similarity scores,

RRF combines document rankings from multiple search methods.

Documents ranked highly by both searches receive higher final rankings.

In [5]:
keyword_rank = {
    "Doc A":1,
    "Doc B":2,
    "Doc C":3
}

semantic_rank = {
    "Doc B":1,
    "Doc A":2,
    "Doc D":3
}

k = 60

scores = {}

for doc, rank in keyword_rank.items():
    scores[doc] = scores.get(doc, 0) + 1 / (k + rank)

for doc, rank in semantic_rank.items():
    scores[doc] = scores.get(doc, 0) + 1 / (k + rank)

sorted(scores.items(), key=lambda x: x[1], reverse=True)

[('Doc A', 0.03252247488101534),
 ('Doc B', 0.03252247488101534),
 ('Doc C', 0.015873015873015872),
 ('Doc D', 0.015873015873015872)]

## Observation

Documents that perform well in both

keyword search

and

semantic search

receive higher final scores.

This often improves retrieval quality without needing to compare score scales directly.

## Advantages

✅ Better retrieval quality

✅ Handles exact terms

✅ Handles semantic meaning

✅ Common in production RAG systems

## Limitations

❌ More complex pipeline

❌ Higher computational cost

❌ Requires combining multiple ranking strategies

## Applications

Hybrid Search is widely used in:

- Enterprise Search
- Financial RAG
- Legal Document Search
- Medical Search
- AI Assistants
- Customer Support Systems

## Comparison

| Method | Best For |
|---------|----------|
| Keyword Search | Exact words, IDs, codes |
| Semantic Search | Meaning and context |
| Hybrid Search | Combining precision and semantic understanding |

##  Think Like a Researcher

Now we can retrieve documents efficiently.

But another important question remains.

How do we know whether our retrieval system is actually good?

Should we measure:

- Precision?
- Recall?
- MRR?
- NDCG?
- Hit Rate?

These evaluation metrics help us compare retrieval systems objectively.

This leads to **Retrieval Evaluation**.